# 08. MCP 고급 서버 활용

> 내가 만든 MCP 서버 외에도 **Context7, Smithery 같은 3rd-party 레지스트리**를 그대로 가져다 쓸 수 있어요. 외부 MCP 서버를 조합해 복합 워크플로우를 짜봐요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **3rd Party MCP 서버**(Context7)를 연결하여 최신 기술 문서를 검색할 수 있어요
2. **복수의 MCP 서버**를 동시에 연결하고 각 서버의 도구를 통합 관리할 수 있어요
3. **MCP 도구와 Tavily 웹 검색**을 결합한 복합 워크플로우를 구축할 수 있어요
4. **Smithery AI** 레지스트리를 활용하여 다양한 3rd Party MCP 서버를 탐색할 수 있어요
5. **복합 작업**(문서 검색 → 웹 검색 → 코드 생성)을 자동화하는 에이전트를 만들 수 있어요

## 사전 지식

- `06-MCP-Server-Basics.ipynb`: FastMCP 서버 생성, stdio/HTTP 전송 방식
- `07-MCP-Agent-Integration.ipynb`: MultiServerMCPClient, create_agent와 MCP 통합
- LangGraph StateGraph 기본 구조 (Part 2 참조)

## 개념 설명: 3rd Party MCP 서버 생태계

MCP(Model Context Protocol)의 가장 강력한 특징은 **서드파티 생태계**예요. 누구나 MCP 서버를 만들고 공유할 수 있고, 이미 수백 개의 유용한 서버들이 공개되어 있어요.

| 서버 | 제공 기능 | 설치 방식 |
|------|----------|----------|
| **Context7** | 라이브러리 최신 문서 검색 | `npx @upstash/context7-mcp` |
| **Brave Search** | 웹 검색 | Smithery 레지스트리 |
| **GitHub** | 저장소 관리 | `npx @modelcontextprotocol/server-github` |
| **Filesystem** | 파일 시스템 접근 | `npx @modelcontextprotocol/server-filesystem` |
| **PostgreSQL** | DB 조회 | `npx @modelcontextprotocol/server-postgres` |

> 🔑 **핵심 개념**: MCP 서버는 **표준 인터페이스**로 동작해요. 마치 스마트폰 **앱 스토어**처럼, Smithery AI에서 원하는 MCP 서버를 찾아 한 줄 명령으로 설치하면 바로 에이전트에서 사용할 수 있어요. 서버가 `stdio`나 `HTTP`로 통신하기만 하면 언어, 프레임워크 무관하게 에이전트에 연결할 수 있어요.

### Smithery AI 레지스트리

[Smithery AI](https://smithery.ai/)는 MCP 서버를 검색하고 설치할 수 있는 공식 레지스트리예요. npm 패키지처럼 서버를 검색하고, 설명과 도구 목록을 확인한 후 바로 사용할 수 있어요.

```bash
# Smithery CLI 설치
npx @smithery/cli install @upstash/context7-mcp --client claude
```

> 💡 **실무 팁**: 새로운 기능이 필요할 때 직접 MCP 서버를 만들기 전에 Smithery에서 검색해보세요. 대부분의 일반적인 도구들은 이미 공개된 서버가 있어요.

## 전체 아키텍처

```mermaid
flowchart TD
    User([사용자 질문]) --> Agent[에이전트<br>create_agent]
    Agent --> Client[MultiServerMCPClient]
    Client --> S1[날씨 서버<br>mcp_server_local.py<br>stdio]
    Client --> S2[시간 서버<br>mcp_server_remote.py<br>HTTP]
    Client --> S3[Context7 MCP<br>@upstash/context7-mcp<br>npx + stdio]
    Client --> S4[웹 검색 서버<br>06_web_search_server.py<br>stdio]
    S1 -- get_weather --> Tools[통합 도구 목록]
    S2 -- get_current_time --> Tools
    S3 -- resolve-library-id<br>query-docs --> Tools
    S4 -- web_search<br>search_news --> Tools
    Tools --> Agent
    Agent --> Result([최종 응답])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class User input
    class Agent,Client process
    class S1,S2,S3,S4 storage
    class Tools,Result output
```

> 💡 **핵심 정리**: 에이전트 입장에서는 도구가 어느 서버에서 왔는지 알 필요가 없어요. `MultiServerMCPClient`가 모든 서버의 도구를 **하나의 목록**으로 통합해주기 때문이에요.

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 및 추적 설정
# ---------------------------------------------------
import os
from dotenv import load_dotenv

# .env 파일에서 API 키들을 로드해요
# OPENAI_API_KEY, TAVILY_API_KEY 등이 필요해요
load_dotenv()

# LangSmith 추적 설정 (선택사항)
# 에이전트의 실행 과정을 LangSmith에서 확인할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-MCP-Advanced"

# 환경 설정 완료

In [ ]:
import asyncio
import concurrent.futures
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 1. 헬퍼 함수 정의

이 노트북 전체에서 사용할 공통 헬퍼 함수들을 먼저 정의할게요.

> 🔑 **핵심 개념**: MCP 클라이언트는 내부적으로 `anyio` 비동기 라이브러리를 사용해요. Jupyter 환경에서는 `nest_asyncio`와 `anyio`의 호환성 문제가 있어서, 위에서 정의한 `run_async()` 헬퍼를 통해 안전하게 실행해요. 프로덕션 코드에서는 `async def main()` 함수 안에서 `await`를 직접 사용하는 것이 일반적이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 기본 모델 설정

이 노트북에서 사용할 기본 LLM을 초기화해요.

> 💡 **실무 팁**: `temperature=0`으로 설정하면 결과가 일관성 있게 나와요. 특히 도구 호출을 해야 하는 에이전트에서는 낮은 temperature를 권장해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Context7 MCP 서버 연결

[Context7](https://github.com/upstash/context7)은 **최신 라이브러리 공식 문서**를 실시간으로 검색할 수 있는 MCP 서버예요.

LLM의 학습 데이터 한계를 극복하는 데 유용해요. 예를 들어:
- LangGraph V1의 최신 API 사용법
- React 19의 변경사항

| Context7 도구 | 용도 |
|--------------|------|
| `resolve-library-id` | 라이브러리 이름으로 ID를 검색해요 |
| `query-docs` | 특정 라이브러리의 문서를 검색해요 |

> ⚠️ **자주 하는 실수**: Context7를 사용하려면 `Node.js`와 `npx`가 설치되어 있어야 해요. `npx --version`으로 확인해요. 없으면 `brew install node` (macOS) 또는 공식 Node.js 사이트에서 설치하세요.

> 💡 **핵심 정리**: `npx -y @upstash/context7-mcp@latest`는 npm 패키지를 설치 없이 바로 실행해요. MCP 서버를 설치하지 않고도 `npx`로 바로 사용할 수 있다는 점이 MCP 생태계의 편의성이에요.

In [ ]:
import subprocess

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 다중 MCP 서버 통합 (3개 서버)

이제 **날씨 서버(stdio) + 시간 서버(HTTP) + Context7(npx)** 세 개를 동시에 연결해볼게요.

```mermaid
flowchart LR
    Client[MultiServerMCPClient] --> W[날씨 서버<br>stdio]
    Client --> T[시간 서버<br>HTTP:8002]
    Client --> C[Context7<br>npx stdio]
    W -- get_weather --> M[통합 도구 4개]
    T -- get_current_time --> M
    C -- resolve-library-id<br>query-docs --> M

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Client process
    class W,T,C storage
    class M output
```

> 🔑 **핵심 개념**: 서버마다 전송 방식이 다를 수 있어요. `stdio`(로컬 프로세스), `streamable_http`(원격 HTTP), `npx`(Node.js 패키지) 모두 동일한 `MultiServerMCPClient`로 관리돼요.

> ⚠️ **자주 하는 실수**: HTTP 방식의 시간 서버는 미리 실행해두어야 해요. 별도 터미널에서 `uv run python servers/02_time_server.py`를 실행하세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 커스텀 워크플로우는 07장에서만 깊게 다룬다

`StateGraph + ToolNode`로 MCP 도구 루프를 직접 만드는 구현은 **05_Agent_Development/07-MCP-Agent-Integration.ipynb**가 소유해요. 이 노트북은 같은 루프를 다시 작성하지 않고, 고급 MCP 서버를 붙일 때 달라지는 **서버 구성, 도구 조합, 도구 큐레이션**에 집중해요.

| 구분 | 담당 범위 |
|------|-----------|
| 07-MCP-Agent-Integration | `ToolNode`, `tools_condition`, `StateGraph`로 MCP 루프 직접 구성 |
| 08-MCP-Advanced-Servers | Context7, 웹 검색, Smithery, 다중 서버, 도구 수 관리 |

> 🔁 **짧은 연결**: 07장의 커스텀 그래프가 필요하면 `ToolNode(tools)` 자리에 여기서 로드한 `context7_tools`, `multi_tools`, `curated_tools`를 넘기면 돼요. 이 장에서는 그 구현을 반복하지 않고, 아래부터 `create_agent` 기반으로 고급 MCP 서버 활용 흐름을 이어갑니다.


## 6. 복합 작업: 최신 문서 검색 + 코드 생성

Context7으로 **최신 LangGraph 문서를 검색**하고, 그 결과를 바탕으로 **코드를 생성**하는 복합 작업을 실행해볼게요.

이것이 MCP의 핵심 사용 사례예요:

```
사용자: "LangGraph로 ReAct 에이전트 만들어줘"
         ↓
에이전트: resolve-library-id("langgraph")  ← Context7
         ↓
에이전트: query-docs(library_id, "react agent")  ← Context7
         ↓
에이전트: [최신 문서 기반으로] 코드 생성
```

> 💡 **핵심 정리**: LLM의 학습 데이터에는 없는 **최신 API**도 Context7를 통해 정확하게 생성할 수 있어요. 이것이 RAG(검색 증강 생성)를 MCP로 구현한 예시예요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. Tavily 웹 검색 + MCP 결합

MCP 도구와 일반 LangChain 도구를 **함께** 사용할 수 있어요. MCP에서 로드한 도구와 Tavily 웹 검색 도구를 합쳐서 하나의 에이전트에 제공할게요.

> 💡 **실무 팁**: `client.get_tools()`가 반환하는 목록은 일반 LangChain `BaseTool` 객체예요. Tavily나 직접 만든 `@tool` 데코레이터 함수와 함께 리스트에 합쳐서 사용할 수 있어요.

> ⚠️ **자주 하는 실수**: Tavily를 사용하려면 `TAVILY_API_KEY`가 `.env`에 있어야 해요. 없으면 `KeyError`가 발생해요.

In [ ]:
from langchain_tavily import TavilySearch

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 실습: 웹 검색 MCP 서버 완성하기

`servers/06_web_search_server.py`를 살펴보고 직접 사용해볼게요.

이 서버는 두 개의 도구를 제공해요:
- `web_search`: 일반 웹 검색
- `search_news`: 뉴스 특화 검색

> 💡 **핵심 정리**: 실습 파일(`06_web_search_server.py`)은 완성된 코드예요. 어떻게 FastMCP로 Tavily를 감싸서 MCP 서버로 만드는지 패턴을 익혀보세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 실습: TODO 블록

아래 TODO 블록을 완성해서 **나만의 3-서버 통합 에이전트**를 만들어보세요!

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3개 서버(Context7 + 웹검색 + 시간)를 통합한
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. Smithery AI 레지스트리 소개

[Smithery AI](https://smithery.ai/)는 MCP 서버를 검색하고 설치할 수 있는 **공식 레지스트리**예요.

### 활용 방법

1. **검색**: https://smithery.ai 에서 원하는 기능 검색
2. **설치 명령 확인**: 각 서버 페이지에서 `npx` 명령을 확인
3. **server_configs에 추가**: 아래와 같이 바로 사용

```python
# Smithery에서 찾은 서버를 바로 추가할 수 있어요
server_configs = {
    # Brave 웹 검색 MCP 서버 (Smithery 레지스트리)
    "brave-search": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "transport": "stdio",
        "env": {"BRAVE_API_KEY": "your-brave-api-key"}
    }
}
```

> 💡 **실무 팁**: MCP 서버를 직접 만들 필요 없이 Smithery에서 검색하면 대부분의 일반적인 도구들을 바로 사용할 수 있어요. GitHub, Slack, Notion, PostgreSQL, Google Drive 등 수백 개의 서버가 공개되어 있어요.

> 🔑 **핵심 개념**: `server_configs`에 `env` 키를 추가하면 해당 서버 프로세스에만 환경변수를 전달할 수 있어요. API 키를 서버별로 분리해서 관리하는 데 유용해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 10. 전체 파이프라인: 4개 서버 통합

지금까지 배운 내용을 종합하여 **4개 서버를 통합한 완성형 에이전트**를 만들어볼게요.

| 서버 | 도구 | 전송 방식 |
|------|------|----------|
| 날씨 서버 | `get_weather` | stdio |
| 시간 서버 | `get_current_time` | HTTP |
| Context7 | `resolve-library-id`, `query-docs` | npx+stdio |
| 웹 검색 | `web_search`, `search_news` | stdio |

> 💡 **핵심 정리**: 이 패턴이 실제 프로덕션에서 MCP를 활용하는 방식이에요. 여러 서버의 도구를 통합하면 에이전트가 **단계별 복합 작업**을 자율적으로 수행할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Context7 MCP 서버**: `npx @upstash/context7-mcp`로 최신 라이브러리 공식 문서를 실시간으로 검색할 수 있어요. LLM의 학습 데이터 한계를 극복하는 데 유용해요
- **3rd Party MCP 생태계**: Smithery AI 레지스트리에서 수백 개의 MCP 서버를 검색하고, `npx` 명령으로 설치 없이 바로 사용할 수 있어요
- **다중 서버 통합**: `MultiServerMCPClient` 딕셔너리에 서버를 추가하기만 하면 도구가 자동으로 통합돼요. `stdio`, `HTTP`, `npx` 방식 모두 동일하게 처리해요
- **복합 워크플로우**: MCP 도구(Context7, 웹 검색)와 일반 LangChain 도구(Tavily)를 리스트에 합쳐서 단일 에이전트에서 사용할 수 있어요
- **커스텀 워크플로우 경계**: `StateGraph + ToolNode` 구현은 07장 소유로 두고, 이 장에서는 Context7·다중 서버·도구 큐레이션 같은 고급 MCP 활용에 집중했어요

---

## 11. 도구 큐레이션

### 왜 도구를 줄여야 하는가

MCP 서버를 여러 개 연결하면 편리하지만, **도구 개수가 늘어날수록 에이전트 성능이 저하**될 수 있어요. MCP는 단순 연결 기술이 아니라, 에이전트에게 어떤 외부 행동을 허용할지 정하는 **하네스 인터페이스**이기도 합니다.

#### 하네스 관점에서 보는 MCP

- **Constrain**: 지금 단계에 필요한 도구만 노출해 선택지를 줄여요
- **Inform**: 도구 이름·description·schema로 모델에게 사용법을 알려줘요
- **HITL**: 외부 시스템을 변경하는 고위험 도구는 승인 절차를 붙여요

#### 모델이 많은 도구를 어려워하는 이유

여러 에이전트 설계 가이드와 산업 사례에서 공통적으로 지적하는 문제는 같아요. 모델에 제공되는 도구 수가 증가할수록 **도구 선택 정확도(tool selection accuracy)** 가 떨어질 수 있습니다. 모델이 어떤 도구를 언제 써야 할지 혼동하는 현상이 생기기 때문이에요.

주요 원인은 두 가지예요:

- **컨텍스트 비용**: 도구 정의(이름 + 설명 + 파라미터 스키마)는 모두 토큰을 소비해요. 도구가 많으면 실제 대화에 쓸 수 있는 컨텍스트 창이 줄어들어요.
- **선택 혼동**: LLM이 유사한 도구들을 구분하지 못하거나 불필요한 도구를 호출해 불필요한 API 비용과 지연이 발생해요.

#### 산업 사례: Harness.io MCP 서버 v2

DevOps 플랫폼 Harness.io는 MCP 서버 v1에서 **130개 이상의 도구**를 노출했어요. 이를 v2에서 **11개의 고수준 도구**로 재설계한 결과:

| 지표 | v1 (130+ 도구) | v2 (11 도구) |
|------|----------------|--------------|
| 컨텍스트 사용 비율 | ~26% | ~1.6% |
| 컨텍스트 창 기준 | 200K 토큰 | 200K 토큰 |
| 도구 선택 오류 | 높음 | 낮음 |

200K 토큰 컨텍스트 창에서 도구 정의만으로 26%를 소비하던 것이 1.6%로 줄었어요. 나머지 컨텍스트를 실제 대화와 추론에 더 많이 활용할 수 있게 됐어요.

> 참고: [Harness.io — MCP Server Redesign](https://www.harness.io/blog/harness-mcp-server-redesign)

#### 설계 원칙

- 도구 하나가 **단일 명확한 책임**을 갖도록 설계
- 비슷한 도구는 **하나의 도구 + 파라미터**로 통합
- 에이전트 워크플로의 각 단계에 **필요한 도구만** 노출

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 큐레이션 전략 3가지

도구 개수를 줄이는 방법은 크게 세 가지예요.

#### 1. 화이트리스트 (Whitelist)

필요한 도구 이름을 명시적으로 지정하고, 나머지는 에이전트에 노출하지 않아요.

```python
# MultiServerMCPClient가 반환한 전체 도구 중 필요한 것만 선택
needed = {"search_docs", "get_weather", "read_file"}
curated_tools = [t for t in tools if t.name in needed]
```

- 장점: 가장 단순하고 예측 가능
- 적합한 상황: 도구 이름을 미리 알고 있는 고정 워크플로

#### 2. 블랙리스트 (Blacklist)

특정 패턴의 도구를 제외하고 나머지를 허용해요.

```python
import re

# "debug_" 로 시작하는 내부 디버그 도구를 제외
curated_tools = [t for t in tools if not re.match(r"debug_", t.name)]
```

- 장점: 새 도구가 추가돼도 자동 포함
- 적합한 상황: 특정 범주(내부 전용, 위험 도구)만 차단하고 싶을 때

#### 3. 워크플로 기반 동적 선택 (Dynamic per Step)

StateGraph의 **각 노드(단계)마다 서로 다른 도구 부분집합**을 LLM에 바인딩해요.

```python
# 검색 단계: 검색 도구만 노출
search_llm = llm.bind_tools([search_tool, docs_tool])

# 생성 단계: 파일 저장 도구만 노출
write_llm  = llm.bind_tools([write_file_tool])
```

- 장점: 각 단계에서 도구 선택 혼동이 최소화
- 적합한 상황: 단계가 명확하게 구분된 파이프라인

> 실무에서는 세 전략을 혼합해요. 예를 들어 블랙리스트로 위험 도구를 먼저 제거하고, 단계별로 화이트리스트를 적용해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import time

from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 워크플로 단계별 동적 큐레이션 패턴 (개념)

실제 프로덕션 에이전트에서는 단일 도구 집합을 고정하는 것보다 **StateGraph 노드마다 필요한 도구만 바인딩**하는 방식이 효과적이에요.

#### 노드별 도구 바인딩 패턴

```python
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# 단계별 도구 분리
SEARCH_TOOLS = [resolve_library_id, get_library_docs, web_search]  # 검색 단계
WRITE_TOOLS   = [write_file, format_code]                           # 생성 단계

# 단계별로 서로 다른 LLM 인스턴스를 바인딩해요
search_llm = llm.bind_tools(SEARCH_TOOLS)
write_llm  = llm.bind_tools(WRITE_TOOLS)

async def search_node(state):
    """검색 도구만 보이는 LLM으로 정보를 수집해요."""
    return {"messages": [await search_llm.ainvoke(state["messages"])]}

async def write_node(state):
    """생성 도구만 보이는 LLM으로 결과를 작성해요."""
    return {"messages": [await write_llm.ainvoke(state["messages"])]}

graph = StateGraph(AgentState)
graph.add_node("search", search_node)
graph.add_node("search_tools", ToolNode(SEARCH_TOOLS))
graph.add_node("write", write_node)
graph.add_node("write_tools", ToolNode(WRITE_TOOLS))

graph.add_edge(START, "search")
graph.add_conditional_edges("search", tools_condition, {"tools": "search_tools", END: "write"})
graph.add_edge("search_tools", "search")
graph.add_conditional_edges("write", tools_condition, {"tools": "write_tools", END: END})
graph.add_edge("write_tools", "write")
```

#### 미들웨어로 단계별 필터링

Part 06에서 배울 `wrap_model_call` 미들웨어를 활용하면, 상태(state)에 현재 단계를 저장하고 모델 호출 시점에 동적으로 도구를 교체할 수 있어요.

```python
from langchain_core.runnables import RunnableConfig

def tool_filter_middleware(model_call):
    async def wrapped(state, config: RunnableConfig):
        phase = state.get("phase", "search")
        tools = SEARCH_TOOLS if phase == "search" else WRITE_TOOLS
        # 단계에 맞는 도구로 교체하고 호출
        return await llm.bind_tools(tools).ainvoke(state["messages"])
    return wrapped
```

> 이 패턴의 핵심은 **LLM이 한 번에 보는 도구 수를 최소화**하는 것이에요. 검색 단계에서는 생성 도구가, 생성 단계에서는 검색 도구가 전혀 보이지 않아요.

### 체크리스트 — MCP 서버 통합 전 점검

MCP 서버를 프로덕션 에이전트에 연결하기 전, 아래 항목을 반드시 확인해요.

#### 도구 수 및 품질

- [ ] 에이전트에 노출되는 도구 개수가 **가급적 20개 이하**인가?
  - `20개`는 공식 한계가 아니라 강의용 경험칙이에요. 실제 기준은 작업 난이도, 모델, 도구 설명 품질에 따라 달라집니다.
  - 20개를 초과하면 먼저 단계별 바인딩, 화이트리스트, 고수준 도구 통합을 검토하세요.
- [ ] 각 도구의 `description`이 **200자 이내**로 간결한가?
  - 불필요하게 긴 설명은 토큰을 낭비하고 모델 혼동을 유발해요.
- [ ] 비슷한 기능의 도구가 **중복**되지 않는가?
  - 예: `search_web`과 `web_search`가 동시에 존재하면 모델이 혼동해요.
- [ ] 도구 이름이 **역할을 명확하게** 표현하는가?
  - 모델은 이름과 설명을 보고 도구를 선택해요. 모호한 이름은 오선택을 유발해요.

#### 워크플로 설계

- [ ] 에이전트 워크플로의 **각 단계마다 어떤 도구가 필요한지** 매핑했는가?
  - 단계별 도구 맵을 문서화하면 큐레이션 기준이 명확해져요.
- [ ] 단계별로 **서로 다른 도구 부분집합**을 바인딩하도록 설계했는가?
- [ ] 도구 선택 오류 발생 시 **폴백(fallback) 로직**이 있는가?

#### 권한 및 승인

- [ ] 읽기 전용 도구와 쓰기/삭제/결제/전송 도구를 구분했는가?
  - 읽기 도구는 자동 실행할 수 있어도, 외부 상태를 바꾸는 도구는 별도 승인 정책이 필요해요.
- [ ] 고위험 MCP 도구에 대해 **confirm / approve / reject** 같은 HITL 흐름을 설계했는가?
- [ ] 도구 description에 "언제 쓰면 안 되는지"가 포함되어 있는가?

#### 비용 및 모니터링

- [ ] `get_tools()`로 로드되는 도구 정의의 **총 토큰 추정치**를 측정했는가?
  - 위 측정 셀을 실행하여 컨텍스트 창 대비 비율을 확인해요.
- [ ] LangSmith 추적으로 **도구 호출 패턴**을 모니터링하고 있는가?
  - 불필요한 도구 호출이 반복되면 큐레이션을 재검토해야 해요.
- [ ] MCP 서버 버전 업데이트 시 **도구 목록 변화**를 자동으로 감지하는가?

---

> 핵심 원칙: "모든 도구를 연결하고 싶은 충동을 참아라."
> 에이전트에게 필요한 도구만 보여주는 것이 성능과 비용 모두를 개선하는 가장 빠른 방법이에요.

## 다음 노트북 예고

다음 `Part 06 / 01-Middleware-Basics.ipynb`에서는 **LangChain V1 미들웨어**를 배워요. `create_agent`의 실행 흐름에 있는 정해진 후크(hook) 지점에 로직을 끼워 넣어 로깅·수정·제어·강제 기능을 구현하는 방법을 다룹니다. 핵심 후크 6가지(`before_agent`, `before_model`, `after_model`, `after_agent`, `wrap_model_call`, `wrap_tool_call`), 편의 데코레이터 `dynamic_prompt`, 대표 내장 미들웨어 6종을 통해 지금까지 만든 에이전트를 **안전하고 관측 가능한 프로덕션 에이전트**로 업그레이드하는 흐름을 배워요.